In [ ]:
# Allows you to use modified modules without rebooting the kernel
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import grangercausalitytests
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller

df = pd.read_json("humidity_2006_2008.json")
display(df.head())

In [ ]:
def grangers_causation_matrix(
    data, variables, test="ssr_chi2test", verbose=False, maxlag=10
):
    """Check Granger Causality of all possible combinations of the Time series.
    The rows are the response variable, columns are predictors. The values in the table
    are the P-Values. P-Values lesser than the significance level (0.05), implies
    the Null Hypothesis that the coefficients of the corresponding past values is
    zero, that is, the X does not cause Y can be rejected.

    data      : pandas dataframe containing the time series variables
    variables : list containing names of the time series variables.
    """
    df = pd.DataFrame(
        np.zeros((len(variables), len(variables))), columns=variables, index=variables
    )
    for c in df.columns:
        for r in df.index:
            test_result = grangercausalitytests(
                data[[r, c]], maxlag=maxlag, verbose=False
            )
            p_values = [round(test_result[i + 1][0][test][1], 4) for i in range(maxlag)]
            if verbose:
                print(f"Y = {r}, X = {c}, P Values = {p_values}")
            min_p_value = np.min(p_values)
            df.loc[r, c] = min_p_value
    df.columns = [var + "_x" for var in variables]
    df.index = [var + "_y" for var in variables]
    return df


GCM = grangers_causation_matrix(df, variables=df.columns)
GCM.to_excel("grangers_causation_matrix.xlsx")

In [ ]:
try:
    params_to_drop = []
    for i in range(3):
        params_to_drop += [f"a_{i}", f"b_{i}", f"p_{i}"]
    params_to_drop += ["Humidity"]
    df.drop(columns=params_to_drop, inplace=True)
except KeyError:
    pass
finally:
    display(df.head())

In [ ]:
nobs = 15
df_train, df_test = df[0:-nobs], df[-nobs:]

# Check size
print(df_train.shape)  # (119, 8)
print(df_test.shape)  # (4, 8)

In [ ]:
def adfuller_test(series, signif=0.05, name="", verbose=False, nostatonly=False):
    """Perform ADFuller to test for Stationarity of given series and print report"""
    r = adfuller(series, autolag="AIC")
    output = {
        "test_statistic": round(r[0], 4),
        "pvalue": round(r[1], 4),
        "n_lags": round(r[2], 4),
        "n_obs": r[3],
    }
    p_value = output["pvalue"]

    def adjust(val, length=6):
        return str(val).ljust(length)

    print(f'    Augmented Dickey-Fuller Test on "{name}"', "\n   ", "-" * 47)
    if verbose:
        # Print Summary
        print(" Null Hypothesis: Data has unit root. Non-Stationary.")
        print(f" Significance Level    = {signif}")
        print(f' Test Statistic        = {output["test_statistic"]}')
        print(f' No. Lags Chosen       = {output["n_lags"]}')
        for key, val in r[4].items():
            print(f" Critical value {adjust(key)} = {round(val, 3)}")

    if p_value <= signif:
        if not nostatonly:
            print(f" => P-Value = {p_value}. Rejecting Null Hypothesis.")
            print(" => Series is Stationary.")
        return True
    else:
        print(f" => P-Value = {p_value}. Weak evidence to reject the Null Hypothesis.")
        print(" => Series is Non-Stationary.")
        return False

In [ ]:
stationary = []
# ADF Test on each column
for name, column in df_train.items():
    stationary += [adfuller_test(column, name=column.name, nostatonly=True)]
    print("\n")

In [ ]:
if not all(stationary):
    # 1st difference
    df_differenced = df_train.diff().dropna()
    # ADF Test on each column of 1st Differences Dataframe
    for name, column in df_differenced.items():
        adfuller_test(column, name=column.name, nostatonly=True)
        print("\n")
else:
    df_differenced = df_train.dropna()

In [ ]:
model = VAR(df_differenced)
max_lag = model.select_order()
# for i in [1,2,3,4,5,6,7,8,9]:
#     try:
#         result = model.fit(i)
#         print('Lag Order =', i)
#         print('AIC : ', result.aic)
#         print('BIC : ', result.bic)
#         print('FPE : ', result.fpe)
#         print('HQIC: ', result.hqic, '\n')
#     except BaseException:
#         print(f"Can't process further. Use lag <= {i-1}")
#         break
# max_lag = i-1

In [ ]:
model_fitted = model.fit(max_lag.selected_orders["aic"])
# model_fitted = model.fit(max_lag)
model_fitted.summary()

In [ ]:
from statsmodels.stats.stattools import durbin_watson

out = durbin_watson(model_fitted.resid)


def adjust(val, length=6):
    return str(val).ljust(length)


for col, val in zip(df.columns, out):
    print(adjust(col), ":", round(val, 2))

In [ ]:
# Get the lag order
lag_order = model_fitted.k_ar

# Input data for forecasting
forecast_input = df_differenced.values[-lag_order:]

# Forecast
fc = model_fitted.forecast(y=forecast_input, steps=nobs)
if not all(stationary):
    df_forecast = pd.DataFrame(fc, index=df.index[-nobs:], columns=df.columns + "_1d")
else:
    df_forecast = pd.DataFrame(fc, index=df.index[-nobs:], columns=df.columns)
display(df_forecast)

In [ ]:
def invert_transformation(
    df_train: pd.DataFrame, df_forecast: pd.DataFrame, second_diff=False
):
    """Revert back the differencing to get the forecast to original scale."""
    df_fc = df_forecast.copy()
    columns = df_train.columns
    for col in columns:
        # Roll back 2nd Diff
        if second_diff:
            df_fc[str(col) + "_1d"] = (
                df_train[col].iloc[-1] - df_train[col].iloc[-2]
            ) + df_fc[str(col) + "_2d"].cumsum()
        # Roll back 1st Diff
        df_fc[str(col) + "_forecast"] = (
            df_train[col].iloc[-1] + df_fc[str(col) + "_1d"].cumsum()
        )
    return df_fc


if not all(stationary):
    df_results = invert_transformation(df_train, df_forecast)
else:
    df_results = df_forecast

In [ ]:
from plotly.express import line

for i, col in enumerate(df.columns):
    # df_results[col+'_forecast'].plot(legend=True, ax=ax).autoscale(axis='x',tight=True)
    data = {"forecast": df_results[col], "actual": df_test[col]}
    fig = line(data)
    fig.update_layout(title=col + ": Forecast vs Actuals")
    fig.show()
    # df_test[col][-nobs:].plot(legend=True, ax=ax);
    # ax.set_title()
    # ax.xaxis.set_ticks_position('none')
    # ax.yaxis.set_ticks_position('none')
    # ax.spines["top"].set_alpha(0)
    # ax.tick_params(labelsize=6)

# plt.tight_layout();

In [ ]:
# # Import StandardScaler
# from sklearn.preprocessing import StandardScaler

# # Instantiate the scaler
# scaler = StandardScaler()

# # Transform data
# scaled_values = scaler.fit_transform(df_diff)

# # Convert to dataframe
# df_scaled = pd.DataFrame(scaled_values, columns=df_diff.columns, index=df_diff.index)
# # Define function for data transformation
# def df_test_transformation(df, test_start_date, scaler):
#     # Apply differencing to make data stationary
#     df_diff = df.diff().dropna()

#     # Scale data using the previously defined scaler
#     df_scaled = pd.DataFrame(
#         scaler.fit_transform(df_diff), columns=df_diff.columns, index=df_diff.index
#     )

#     # Select only the data that belongs to the testing set
#     df_test_processed = df_scaled[df_scaled.index > test_start_date]

#     return df_test_processed


# # Define function for inverting data transformation
# def df_inv_transformation(df_processed, df, scaler):
#     # Invert StandardScaler transformation
#     df_diff = pd.DataFrame(
#         scaler.inverse_transform(df_processed),
#         columns=df_processed.columns,
#         index=df_processed.index,
#     )

#     # Invert differenting
#     df_original = df_diff.cumsum() + df[df.index < df_diff.index[0]].iloc[-1]

#     return df_original


# # Apply function to our data
# df_test_processed = df_test_transformation(df_roll, df_scaled.index[-1], scaler)
# # Get optimal lag order based on the four criteria
# optimal_lags = model.select_order()

# print(f"The optimal lag order selected: {optimal_lags.selected_orders}")